In [14]:
from pageindex import PageIndexClient
import pageindex.utils as utils

from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

In [15]:
from google import genai

gemini_api_key = os.getenv("GEMINI_API_KEY")
def call_llm(prompt):
    client = genai.Client(api_key=gemini_api_key)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text

In [16]:
pdf_path ="/Users/nikitababbar/Desktop/vectorless rag/sample.pdf"
doc_id = pi_client.submit_document(pdf_path)["doc_id"]
print('Document Submitted:', doc_id)

Document Submitted: pi-cmm60wn5w03ej0mo930wuzbv1


In [23]:
tree = pi_client.get_tree(doc_id, node_summary=True)['result']
print('Simplified Tree Structure of the Document:')
utils.print_tree(tree)

Simplified Tree Structure of the Document:
[{'title': 'Preface', 'node_id': '0000', 'summary': 'The text discusses the American health c...'},
 {'title': 'Health and the Demand for Health Care',
  'node_id': '0001',
  'prefix_summary': '# Health and the Demand for Health Care\n...',
  'nodes': [{'title': 'Demand for Health',
             'node_id': '0002',
             'summary': '## Demand for Health\n\nPeople demand heal...'},
            {'title': 'The Production of Health',
             'node_id': '0003',
             'summary': 'The text discusses health as a product o...'},
            {'title': 'Trends in Health Spending',
             'node_id': '0004',
             'summary': 'The text details the significant upward ...'},
            {'title': 'Trends in Life Expectancy',
             'node_id': '0005',
             'summary': 'The text discusses trends in US life exp...'},
            {'title': 'Trends in Health Insurance Coverage',
             'node_id': '0006',
          

In [19]:
import json

query = "What are the conclusions in this document?"

tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format (strictly adhere to this format): 
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = call_llm(search_prompt)

In [20]:
node_map = utils.create_node_mapping(tree)

# Debug: Print the raw response
print("Raw LLM Response:")
print("-" * 80)
print(tree_search_result)
print("-" * 80)

# Clean the response - remove markdown code blocks if present
cleaned_result = tree_search_result.strip()
if cleaned_result.startswith("```"):
    # Remove opening code fence
    lines = cleaned_result.split('\n')
    if lines[0].startswith("```"):
        lines = lines[1:]
    # Remove closing code fence
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    cleaned_result = '\n'.join(lines)

tree_search_result_json = json.loads(cleaned_result)

print('\nReasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Raw LLM Response:
--------------------------------------------------------------------------------
```json
{
    "thinking": "The question asks for the 'conclusions' in this document. The most direct and relevant node for this request is the one explicitly titled 'Conclusion'. Node '0018' is titled 'Conclusion' and its summary clearly outlines the main takeaways and proposed solutions discussed in the document, which are essentially the conclusions drawn by the author.",
    "node_list": ["0018"]
}
```
--------------------------------------------------------------------------------

Reasoning Process:
The question asks for the 'conclusions' in this document. The most direct and relevant node for this
request is the one explicitly titled 'Conclusion'. Node '0018' is titled 'Conclusion' and its
summary clearly outlines the main takeaways and proposed solutions discussed in the document, which
are essentially the conclusions drawn by the author.

Retrieved Nodes:
Node ID: 0018	 Page: 18	 

In [21]:
node_list = json.loads(cleaned_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

# Conclusion

The health care system in the United States has helped improve the health and well-being of
Americans. As health care costs continue to rise, enormous opportunities exist to increase the value
of health care and improve health insurance coverage. Addressing these fundamental problems and
fulfilling the potential of our health care system will require innovative polices to help Americans
get the care that best meets their needs, and to create an environment that rewards high-quality,
efficient care. While Federally sponsored health insurance for the most vulnerable Americans through
Medicare, Medicaid, and SCHIP remains a priority, private markets offer the best opportunities for
controlling costs and providing innovative policies to enhance efficiency, quality, and access.
Efficiency of health spending would be improved if tax code reforms were enacted. Reforms could
level the playing field between employer-provided and individual health insurance, thu

In [22]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer =  call_llm(answer_prompt)
utils.print_wrapped(answer)

Generated Answer:

The document concludes that:
* The U.S. healthcare system has improved health, but rising costs present opportunities to increase
its value and improve health insurance coverage.
* Innovative policies are required to help Americans get appropriate care and to reward high-
quality, efficient care.
* While Federally sponsored health insurance for the vulnerable is a priority, private markets offer
the best opportunities for controlling costs and providing innovative policies.
* Tax code reforms could improve health spending efficiency, level the playing field between
insurance types, boost coverage, and encourage higher deductible plans.
* Addressing adverse selection can make insurance markets more competitive, promoting innovation,
choice, access, and efficiency.
* Health care quality can be improved by increasing transparency of information and tying
reimbursement to provider performance.
